# 传统神经网络相关概念——以ResNet50为例

ResNet-50是一个经典的卷积神经网络(CNN)，主要用于图像识别，也是深度学习里非常具有代表性的基础模型，其核心就是不断通过CNN提取 feature。torchvision是PyTorch生态里专门服务于计算机视觉的库。
首先导入基本的ResNet50模型，其中resnet50是构造函数，它内部知道ResNet50的结构。ResNet50_Weights是预训练权重版本，它告诉torchvision，等会构造模型的时候，请使用这套 pretrained weights。构造函数中，先根据resnet50的定义创建网络结构，再把刚才导入的预训练权重填进去。此时import时是下载的主要内容不是“神经网络代码”，而是已经训练好的参数张量。ResNet-50整体大约有两千多万个参数。model.eval是把模型切换成推理模式，这个和GPT-2里面的内容类似。

ResNet系列的主要特点是，假设需要学习的目标函数$y = T(x)$，普通传统的神经网络可能会去学习$F(x) \approx T(x)$。这种情况下，假设$T(x) = x$，需要付出一定的学习成本。但是ResNet学习的是$ T(x) \approx F(x) + x$，这种情况下实际上参与学习的是$T(x) - x$也就是x需要改变多少。当$T(x) = x$时$F(x)$只需要维持为0即可。
另外从梯度的角度也具有优势，传统的神经网络$y = F(x)$，对应的偏导数$\frac{\partial y}{\partial x} = \frac{\partial F}{\partial x}$。当这一层梯度很小的时候，模型可能不容易收敛；但是对于ResNet，$y = F(x) + x$，对应的剃度$\frac{\partial y}{\partial x} = \frac{\partial F}{\partial x} + 1$，这个1保证了梯度不会太小，会使训练更容易收敛。

In [1]:
from PIL import Image
from torchvision.models import resnet50, ResNet50_Weights
import torch
import sys
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", colorize=True)

1

In [2]:
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

这里对上面的输出进行详细说明，前面的Conv → BatchNorm → ReLU → MaxPool做的是不同的东西。
+ conv1的Conv2d是Convolution卷积相关内容，2D就是处理二维图像。kernel_size是卷积维度，这里的3是输入channel数，因为RGB所以是3；64指的是ResNet第一层有64个不同的卷积核，输出channel是64，会产生64张feature map。stride说的是卷积核每次移动横向和纵向各2个像素。padding说的是在图片周围补充3曾像素，通常补0，这样可以让边缘的像素也被充分处理。
+ ReLU全称是Rectified Linear Unit，线性修正单元，其在自变量<0的时候与x轴平行，在自变量>0的时候与y=x重合。ReLU本身是分段线性的。每一层都加ReLU则可以拟合出复杂的函数形状。
+ bn代表batch normalization是批量归一化，归一化到例如均值0，标准差1之后，再学习缩放系数对其进行线性变换。因此有了归一化，上一步的bias就失去了意义，选择False即可。
+ maxpool指的是Max Pooling，最大池化学，即在一个区域内只留下最大的一个数。用于降低数据量，同时可以保留最显著的特征。
+ 其中的layer1是按照顺序执行的一个模块，其中的每一个Bottleneck会顺序执行。Bottleneck的含义是，每一个残差块中，都会执行先降维再升维的操作来节省计算量
+ 每一个layer中，就在执行降维、卷积、升维然后ReLU的操作。其中部分层有的downsample是将shortcut路径上的维度与bottle neck路径上的维度统一，才能够想嫁。

首先加载图片以及使用官方的预处理过程。首先使用PIL(python image library)读取图片并且使用RGB编码，而后preprocess指的是ResNet50权重对应的预处理流程，包括诸如resize、center crop、转成tensor、normalize的过程，最终变成模型需要的[3,224,224]维度的tensor，因为希望推理时的数据分布和训练时基本保持一致。

In [3]:
image = Image.open("assets/dog.jpeg").convert("RGB")

preprocess = weights.transforms()
input_tensor: torch.Tensor = preprocess(image)
logger.info(f"input tensor size before unsqueeze = {input_tensor.size()}")
# 增加 batch 维度
input_batch = input_tensor.unsqueeze(0)
logger.info(f"input tensor size after unsqueeze = {input_tensor.size()}")

2026-08-12 20:27:01.616 | INFO     | __main__:<module>:5 - input tensor size before unsqueeze = torch.Size([3, 224, 224])
2026-08-12 20:27:01.616 | INFO     | __main__:<module>:8 - input tensor size after unsqueeze = torch.Size([3, 224, 224])


而后进行推理过程以及logits的处理，与GPT-2类似，这里的logits则是ResNet50的类别对应的分数，其softmax之后输出前三项得分及其对应的种类。可以看到其预测结果。

In [4]:
with torch.no_grad():
    logits = model(input_batch)
logger.info(f"logits shape = {logits.shape}")

probs = torch.softmax(logits[0], dim=0)
top_probs, top_ids = torch.topk(probs, 3)
categories = weights.meta["categories"]
for prob, idx in zip(top_probs, top_ids):
    print(f"{categories[idx]}: {prob.item():.4f}")

2026-08-12 20:27:01.654 | INFO     | __main__:<module>:3 - logits shape = torch.Size([1, 1000])
toy poodle: 0.5296
miniature poodle: 0.0604
Blenheim spaniel: 0.0029
